# IceCube Group Notebook: Multi-Messenger Astronomy
## Finding a Neutrino's Cosmic Source

**Vanderbilt Programs for Talented Youth | Instructor: Jennifer James, Ph.D. Candidate**

---

### The challenge

IceCube is a cubic-kilometer detector buried in the Antarctic ice, built to catch high-energy neutrinos arriving from space. Neutrinos barely interact with anything, which is exactly why they are useful: they travel in a straight line from wherever they were produced, unbent by magnetic fields and unabsorbed by dust or gas, unlike charged cosmic rays or even gamma rays.

But that same property makes finding their source hard. Most of the neutrinos IceCube sees are background, made in Earth's own atmosphere by ordinary cosmic ray showers, with no connection to any distant source.

On September 22, 2017, IceCube detected a single very high-energy neutrino (event IceCube-170922A) and sent out an automatic alert with its reconstructed direction. Telescopes around the world pointed at that patch of sky, and Fermi-LAT and MAGIC found that a known blazar, TXS 0506+056, was in the middle of an unusually bright gamma-ray flare at that exact time and direction. This was the first strong evidence connecting a single high-energy neutrino to a specific astrophysical source, a real multi-messenger detection: neutrino plus light, from the same object.

In this notebook, you will build a simplified version of the same kind of analysis: given a sky full of background neutrinos and one alert event, how do you decide whether it really points back to a flaring source, or whether it is just a coincidence?

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(170922)  # the date of the real alert, 2017-09-22
print("Libraries loaded!")

---

## Part 1: Simulating a Patch of Sky

We will work in a small patch of sky centered on the blazar's known position, using offsets in degrees (delta RA, delta Dec) rather than full sky coordinates.

- **Background neutrinos**: produced in Earth's atmosphere, arriving from random directions with no preference for the blazar's location, and at random times over the whole observation window
- **The alert event**: IceCube's real angular reconstruction uncertainty for a track-like event is roughly half a degree; we place the true direction exactly at the blazar and smear it by that resolution
- **The blazar's gamma-ray brightness**: mostly quiet, with one flare episode centered on the alert time

In [ ]:
# ── Observation window and field of view ──────────────────────────────────
T_total   = 7.0     # years of IceCube data searched (2012-2019 era, full detector)
fov_deg   = 5.0      # half-width of the sky patch we search, in degrees

# ── Background neutrino population ─────────────────────────────────────────
N_bg = 4000
bg_dRA  = np.random.uniform(-fov_deg, fov_deg, N_bg)
bg_dDec = np.random.uniform(-fov_deg, fov_deg, N_bg)
bg_time = np.random.uniform(0, T_total, N_bg)

# ── The alert event ─────────────────────────────────────────────────────────
angular_resolution = 0.5   # degrees, typical for a through-going track event
alert_dRA  = np.random.normal(0, angular_resolution)
alert_dDec = np.random.normal(0, angular_resolution)
alert_time = 5.72   # years into the window (places it near day 2100, our "Sept 2017")

print(f"Simulated sky patch: {2*fov_deg} deg x {2*fov_deg} deg, {T_total} years")
print(f"Background events:   {N_bg}")
print(f"Alert event offset from blazar: dRA={alert_dRA:.2f} deg, dDec={alert_dDec:.2f} deg")
print(f"Alert event time:    {alert_time:.2f} years into the window")

---

## Part 2: The Blazar's Gamma-Ray Light Curve

Blazars are not steady gamma-ray sources. They flare. We model a single flare episode as a Gaussian bump in brightness centered on the alert time.

In [ ]:
def gamma_ray_flux(t, baseline, flare_amp, flare_time, flare_width):
    """Quiescent baseline flux plus one Gaussian flare."""
    flare = flare_amp * np.exp(-0.5 * ((t - flare_time) / flare_width) ** 2)
    return baseline + flare

t_axis      = np.linspace(0, T_total, 2000)
baseline    = 1.0
flare_amp   = 6.0
flare_width = 0.08   # years, roughly a month-scale flare

flux = gamma_ray_flux(t_axis, baseline, flare_amp, alert_time, flare_width)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(t_axis, flux, color='steelblue', linewidth=2, label='TXS 0506+056 gamma-ray flux (model)')
ax.axvline(x=alert_time, color='firebrick', linestyle='--', linewidth=2, label='Neutrino alert arrival time')
ax.set_xlabel('Time (years into observation window)', fontsize=12)
ax.set_ylabel('Relative gamma-ray flux', fontsize=12)
ax.set_title('Blazar gamma-ray light curve with one flare episode', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("The neutrino arrived right at the peak of a gamma-ray flare.")
print("By itself, is that enough to claim the neutrino came from the blazar?")

---

## Part 3: Does the Alert Event Point Back to the Blazar?

Time coincidence alone is not enough; thousands of background neutrinos arrive during any given month. We also need the alert event's *direction* to match the blazar's position, well within the detector's angular resolution.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(bg_dRA, bg_dDec, s=8, color='gray', alpha=0.4, label='Background neutrinos')
ax.scatter([0], [0], marker='*', s=400, color='gold', edgecolor='black',
           linewidth=1, label='TXS 0506+056 (blazar position)', zorder=5)
ax.scatter([alert_dRA], [alert_dDec], s=200, color='firebrick', marker='o',
           edgecolor='black', linewidth=1.5, label='Alert neutrino (IceCube-170922A-like)', zorder=6)
circle = plt.Circle((0, 0), angular_resolution, color='gold', fill=False,
                     linestyle='--', linewidth=1.5, label='1 sigma angular resolution')
ax.add_patch(circle)
ax.set_xlabel('Delta RA (degrees)', fontsize=12)
ax.set_ylabel('Delta Dec (degrees)', fontsize=12)
ax.set_title('Sky map centered on the blazar', fontsize=12)
ax.set_xlim(-fov_deg, fov_deg)
ax.set_ylim(-fov_deg, fov_deg)
ax.set_aspect('equal')
ax.legend(fontsize=9, loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

angular_sep = np.sqrt(alert_dRA**2 + alert_dDec**2)
print(f"Angular separation between alert event and blazar: {angular_sep:.2f} degrees")
print(f"Detector angular resolution (1 sigma): {angular_resolution:.2f} degrees")
print(f"Separation in units of angular resolution: {angular_sep/angular_resolution:.2f} sigma")

---

## Part 4: Estimating the Chance of a Coincidence

Even a background neutrino could, by pure chance, land close to the blazar during the flare. We estimate how often that would happen, given how many background events we recorded and how large our search window is.

In [ ]:
# ── Position probability: solid angle within the search radius / total field of view ──
search_radius = 1.0   # degrees, a typical signal search cone
area_search = np.pi * search_radius**2
area_fov    = (2*fov_deg)**2
p_position  = area_search / area_fov

# ── Time probability: flare active window / total observation time ────────────
flare_active_window = 3 * flare_width   # roughly the effective flare duration, in years
p_time = flare_active_window / T_total

# ── Chance coincidence probability and expected number of chance events ────────
p_chance = p_position * p_time
expected_chance_events = N_bg * p_chance

print(f"Probability a single background event lands in the search cone: {p_position:.5f}")
print(f"Probability a single background event lands in the flare window: {p_time:.5f}")
print(f"Combined chance probability per background event:  {p_chance:.6f}")
print(f"Expected number of chance coincidences from {N_bg} background events: {expected_chance_events:.3f}")
print()
if expected_chance_events < 0.05:
    print("We expect far less than one chance coincidence over the whole dataset.")
    print("Finding one real event that matches both position and time is a meaningful signal.")
else:
    print("Chance coincidences are not rare enough here to draw a strong conclusion alone.")

---

## Part 5: Why Detector Volume Matters

Neutrinos interact so weakly that IceCube needs an enormous volume of material, a full cubic kilometer of Antarctic ice instrumented with light sensors, just to catch a handful of astrophysical neutrinos per year. This is the neutrino equivalent of the tracker-radius and calorimeter-thickness tradeoffs your classmates have run into: a rare, weakly-interacting signal forces the detector to be big, not necessarily precise in the way a small tracker layer would be.

In [ ]:
# ── Toy model: expected detected neutrinos scales with detector volume ────────
volumes_km3   = np.array([0.001, 0.01, 0.1, 1.0, 5.0])   # candidate detector volumes
rate_per_km3  = 8   # toy astrophysical neutrino detections per km^3 per year, at these energies

expected_per_year = rate_per_km3 * volumes_km3

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(volumes_km3, expected_per_year, marker='o', markersize=8,
        color='steelblue', linewidth=2)
ax.axvline(x=1.0, color='firebrick', linestyle='--', linewidth=1.5,
           label='IceCube (1 km^3, current)')
ax.set_xlabel('Detector volume (km^3)', fontsize=12)
ax.set_ylabel('Expected astrophysical neutrinos / year (toy model)', fontsize=12)
ax.set_title('Why neutrino telescopes need to be enormous', fontsize=12)
ax.set_xscale('log')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

print("A next-generation detector like IceCube-Gen2 is planned at roughly 8 km^3.")
print("Scaling up volume is the main lever for catching more astrophysical neutrinos,")
print("since the interaction probability per neutrino cannot be improved; it is fixed by")
print("the weak interaction cross section.")

---

## Your Group's Checkpoint Questions

1. In Part 3, why is angular resolution alone not enough to identify a source? Why did we need both a position match (Part 3) and a time match (Part 2) before drawing any conclusion in Part 4?

2. In the real 2017 event, an independent archival search later found an earlier excess of neutrinos from the direction of TXS 0506+056 between 2014 and 2015, during a period when the blazar was not flaring in gamma rays at the time. What does this tell you about the limits of using gamma-ray flares alone to identify neutrino sources?

3. Why can neutrinos travel in a straight line back to their source, while charged cosmic rays generally cannot? What does this mean for using neutrinos specifically for source-pointing, compared to other messengers?

4. In Part 5, why does increasing detector volume help, when it does not change the interaction probability of any single neutrino? What physical quantity is actually being increased?

5. From your subsystem checklist, does your IceCube-based detector design change at all based on this notebook? Specifically, what would you prioritize to also enable *transient* alerts, sending out a real-time direction as soon as one high-energy event is seen, rather than only analyzing data after the fact?